In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
import time
from birddog.core import (
    Archive,
    ArchiveWatcher,
    PageUpdateManager,
    PageLRU,
    )
from birddog.wiki import get_all_pages, mw_read_page, canonicalize_title

2025-07-10 14:57:12,750 [INFO] Using Google Cloud translation API (credentials file:/Users/jbrandt/code/birddog/google-cloud-translate-key.json)
2025-07-10 14:57:12,756 [INFO] Using local folder /Users/jbrandt/code/birddog/.cache for storage.


In [3]:
titles = [ 
    "Архів:ДАПО/Р-9126/9",
    "Архів:ДАЖО/1/1",
    "Архів:ДАК/312/1",
    "Архів:ДАЖО/1/74",
    "Архів:Лука_Мала",
    "Архів:ЦДІАК/1/1",
    "Архів:ДАХО/Д",
    "Архів:ДАПО/Р/1–1000",
    "Архів:ЦДІАК/28",
    "Архів:ЦДІАК/28/1",
    "Архів:ДАЖО/1",
    "Архів:ДАК/Р-352",
    "Архів:Архівний відділ виконавчого комітету Кременчуцької міської ради/Р",
    "Архів:ДАЖО/Д",
    "Архів:ДАДнО/Р-6478/2", 
    "Архів:ДАЖО/752", 
    "Архів:ДАКрО/225/1/25", 
    "Архів:ДАКрО/225", 
    "Архів:ДАСО/Р", 
    "Архів:ДАХмО/К", 
    "Архів:ДАКрО/225/1/144а", 
    "Архів:ДАПО/978/1",
    "Архів:ДАПО/Р",
    "Архів:ДАХмО/Р-6193",
    "Архів:ДАКрО/П-5907/2Р",
    "Архів:ДАОО/Р-8085/1",
    "Архів:ДАКрО/225/1",
    "Архів:ДАПО/1072/1/1",
    "Архів:ДАПО/978/1/135",
    "Архів:ДАПО/978",
    "Архів:ДАКО/Р-5634/1/3092",
    ]

In [4]:
manager = PageUpdateManager()
manager.heartbeat()

2025-07-10 14:57:16,364 [INFO] PageUpdateManager: checking for page updates...
2025-07-10 14:57:16,576 [INFO] fetch_url: 1 requests in last 60s → 0.02 req/s
2025-07-10 14:57:16,577 [INFO] PageUpdateManager: finished update check...


In [5]:
manager._title_index.lookup("Архів:ДАПО/Р/9126")

2025-07-10 14:57:55,845 [INFO] TitleIndex.lookup(Архів:ДАПО/Р/9126)
2025-07-10 14:57:56,065 [INFO] fetch_url: 2 requests in last 60s → 0.03 req/s


('DAPO', 'R', '9126', '', '')

In [ ]:
for title in titles[:1]:
    address = manager._title_index.lookup(title)
    page = manager._lru.lookup(*address)
    print(f"{title}: {address}, {page.title}")
    assert canonicalize_title(title) == canonicalize_title(page.title)

In [ ]:
def select_parent_archive(archive_root, fond_id, ti=manager._title_index):
    parent_archive = None
    known_child = False
    if not archive_root in ti._archives:
        raise ValueError(f"Unknown archive root: {archive_root}")
    for archive_address in ti._archives[archive_root]:
        archive = ti._lru.lookup(*archive_address)
        #print(archive.title)
        if fond_id.upper().startswith(archive.subarchive["uk"]):
            parent_archive = archive_address
            known_child = fond_id in archive.child_ids
            break
        elif archive_address[1] == "D" or len(ti._archives[archive_root]) == 1:
            parent_archive = archive_address
            known_child = fond_id in archive.child_ids
    return parent_archive, known_child

In [ ]:
def test_title(title, ti=manager._title_index):
    try:
        address = ti.lookup(title)
        return "known"
        #print("known:", title, address)
    except ValueError:
        title = canonicalize_title(title)
        title_split = title.split("/")
        if len(title_split) > 1:
            try:
                parent_archive, known_fond = select_parent_archive(*title_split[:2], ti)
                if not known_fond:
                    #print("need to adopt:", parent_archive, title)
                    return "adopt"
                else:
                    #print("something unexpected:", title)
                    return "unexpected"
            except ValueError:
                #print("unknown archive root:", title)
                return "unknown_archive"
        else:
            #print("unknown archive:", title)
            return "unknown_archive"

In [ ]:
test_title("Архів:ДАЧкО/5899/1")

In [ ]:
adoptees = [title for title in manager._tracker._mod_dates.keys() if test_title(title) == "adopt"]

In [ ]:
len(adoptees)

In [ ]:
adoptees[:20]

In [ ]:
test_result = {title: test_title(title) for title in manager._tracker._mod_dates.keys()}

In [ ]:
[title for title, result in test_result.items() if result == "unknown_archive"]

In [ ]:
orphans = []
for adoptee in adoptees:
    title_split = adoptee.split("/")
    if len(title_split) > 2:
        if test_title("/".join(title_split[:2])) != "adopt":
            orphans.append(adoptee)

In [ ]:
items = manager._tracker._mod_dates

In [ ]:
len(items.keys())

In [ ]:
suspects = [k for k in items.keys() if k.startswith("Архів:ДАПО") and "9126" in k and "Р-9126" not in k]

In [ ]:
from birddog.wiki import batch_page_exists

In [ ]:
batch_page_exists(suspects)